# 🧪 معمل التجارب: أساسيات التقطيع الدلالي المتقدم Semantic Chunking (Lecture 37)
أهلاً بك في معملك التطبيقي للمرحلة الرابعة! 🚀

**فكرة المعمل ببساطة:**
في كل الدروس السابقة، كنا نستخدم أداة `RecursiveCharacterTextSplitter` التي تقطع النصوص بالمسطرة والعدد (مثلاً 500 حرف).
لكن التقطيع بالعدد يعاني من **كارثة حقيقية**:
لو انتهى الحجم في منتصف فكرة معينة، يتم تمزيق الجملة! أو يتم دمج موضوعين ليس بينهما أي علاقة (مثل دمج كلام عن الذكاء الاصطناعي مع كلام عن برج إيفل في نفس القطعة)، مما يشوه معنى المتجهات في الـ `RAG`.

**الحل هو التقطيع الدلالي (`Semantic Chunking`):**
بدل عد الأحرف، نقيس **المعنى الرياضي**؛ ندمج الجمل طالما تتحدث عن نفس الفكرة، وأول ما يتغير الموضوع، نقطع فوراً ونبدأ قطعة جديدة!

في هذا المعمل ستتعلم وتختبر عملياً:
1. تجربة مشكلة التقطيع العادي بالأحرف ورؤية تشوه السياق.
2. خطوات خوارزمية التقطيع الدلالي الأربعة خطوة بخطوة.
3. حساب تشابه جيب التمام (`Cosine Similarity`) بين الجمل المتجاورة.
4. تحديد عتبة الفصل (`Threshold`) وفصل القطع عند الهبوط الدلالي (`Semantic Drop`).
5. رسم منحنى التشابه لرؤية نقاط الانفصال بالعين المجردة!

> 💡 **طريقة التدريب في المعمل:**
> اقرأ الشرح المبسط ⬅️ شوف الكود المرجعي الجاهز ⬅️ واكتب الكود وجربه بإيدك في الخلية الفارغة المخصصة لك تحت كل خطوة!

### 1. استيراد المكتبات وتجهيز النص التجريبي
**الفكرة ببساطة:**
بنستورد مكتبات لانج تشين ونموذج التضمين المجاني من `Hugging Face`، ونجهز نفس النص النموذجي الذي شرحه المحاضر في الفيديو:
نص يحتوي على فكرتين مختلفتين تماماً: فكرة عن إطار عمل `LangChain` وفكرة عن مدينة باريس وفرنسا.

In [ ]:
# 1. import standard libraries
import re
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

# sample corpus from lecture containing two distinct semantic contexts
sample_text = (
    "LangChain is a framework to build RAG applications. "
    "LangChain can also be used to build Generative AI applications and multi-agent workflows. "
    "Paris is the capital of France. "
    "The Eiffel Tower is located in Paris, France and is a popular destination."
)

# initialize embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("✅ Libraries imported successfully!")
print("📄 Sample Document:")
print(sample_text)

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب تجهيز النص واستيراد نموذج التضمين بنفسك هنا):



### 2. تجربة التقطيع العادي وملاحظة المشكلة (`RecursiveCharacterTextSplitter`)
**الفكرة ببساطة:**
تعال نجرب نقطع هذا النص بالتقطيع العادي بحجم 110 أحرف مثلاً.
ستلاحظ أن الساطور الأعمى سيقطع جمل `LangChain` ويخلط ذيلها مع جملة `Paris is the capital of France` في نفس القطعة!

In [ ]:
# split using fixed-character size
fixed_splitter = RecursiveCharacterTextSplitter(chunk_size=110, chunk_overlap=15)
fixed_chunks = fixed_splitter.split_text(sample_text)

print(f"⚠️ Fixed-size Splitter produced {len(fixed_chunks)} chunks:\n")
for idx, chunk in enumerate(fixed_chunks, start=1):
    print(f"--- Chunk [{idx}] ({len(chunk)} characters) ---")
    print(f"'{chunk}'\n")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب تشغيل التقطيع العادي بحجم صغير ولاحظ تداخل الموضوعات هنا):



### 3. الخطوة الأولى: تقسيم المستند إلى جمل فردية (`Document Segmentation`)
**الفكرة ببساطة:**
أول خطوة في التقطيع الدلالي هي تفكيك النص إلى وحداته الأساسية (جمل منفصلة) باستخدام علامات الترقيم (النقطة، علامة الاستفهام، علامة التعجب).

In [ ]:
# step 1: split text into individual sentences using regex
# split on full stops followed by a space
sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", sample_text) if s.strip()]

print(f"✅ Step 1: Extracted {len(sentences)} individual sentences:\n")
for idx, sentence in enumerate(sentences, start=1):
    print(f"  Sentence [{idx}]: '{sentence}'")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب تقسيم النص إلى جمل فردية باستخدام regex بنفسك هنا):



### 4. الخطوة الثانية: حساب المتجهات لكل جملة (`Sentence Embeddings`)
**الفكرة ببساطة:**
نحول كل جملة من الجمل الأربعة إلى متجه عددي يعبر عن معناها باستخدام نموذج التضمين، ونفحص شكل المصفوفة الناتجة.

In [ ]:
# step 2: compute embedding vector for each individual sentence
sentence_vectors = [np.array(embeddings.embed_query(s)) for s in sentences]

print(f"✅ Step 2: Computed {len(sentence_vectors)} dense vectors.")
print(f"📏 Dimension of each vector: {len(sentence_vectors[0])} dimensions.")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب حساب التضمين لكل جملة وفحص أبعاد المتجه بنفسك هنا):



### 5. الخطوة الثالثة: قياس تشابه الجمل المتجاورة (`Pairwise Cosine Similarity`)
**الفكرة ببساطة:**
الآن نقيس الزاوية الدلالية بين كل جملة والجملة التي تليها مباشرة:
- بين الجملة 1 و 2 (كلاهما عن لانج تشين) ⬅️ تتوقع تشابهاً عالياً جداً!
- بين الجملة 2 و 3 (لانج تشين مقابل باريس) ⬅️ هبوط حاد في التشابه (Semantic Drop)!
- بين الجملة 3 و 4 (باريس وبرج إيفل) ⬅️ عودة للتشابه العالي.

In [ ]:
# step 3: calculate cosine similarity between consecutive adjacent sentences
def cosine_similarity(v1, v2):
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

similarities = []
for i in range(len(sentence_vectors) - 1):
    sim = cosine_similarity(sentence_vectors[i], sentence_vectors[i + 1])
    similarities.append(sim)
    print(f"🔍 Similarity between Sentence [{i+1}] and [{i+2}]: {sim:.4f}")

print("\nلاحظ الهبوط المفاجئ في التشابه بين الجملة 2 والجملة 3 (نقطة تغير الموضوع)!")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب كتابة دالة حساب التشابه وفحص الأرقام بنفسك هنا):



### 6. الخطوة الرابعة: دمج الجمل وفصلها عند نقطة التحول (`Threshold Breakpoint`)
**الفكرة ببساطة:**
نحدد عتبة تشابه (مثلاً `threshold = 0.5`):
- إذا كان التشابه بين الجملتين أكبر من أو يساوي 0.5 ⬅️ ندمجهما في نفس الـ `Chunk`.
- إذا هبط التشابه تحت 0.5 ⬅️ نقطع فوراً ونبدأ `Chunk` جديد!

In [ ]:
# step 4: chunk formation based on similarity threshold
threshold = 0.5
semantic_chunks = []
current_chunk = [sentences[0]]

for i, sim in enumerate(similarities):
    if sim >= threshold:
        # high semantic similarity: merge into current chunk
        current_chunk.append(sentences[i + 1])
    else:
        # low semantic similarity (topic change): cut and finalize chunk
        semantic_chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i + 1]]

# append final chunk
if current_chunk:
    semantic_chunks.append(" ".join(current_chunk))

print(f"🎯 Semantic Chunking successfully produced {len(semantic_chunks)} pure chunks:\n")
for idx, chunk in enumerate(semantic_chunks, start=1):
    print(f"=== Semantic Chunk [{idx}] ===")
    print(f"'{chunk}'\n")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب دمج وفصل الجمل حسب الـ threshold بنفسك هنا):



### 7. رؤية الانخفاض الدلالي برسم بياني واضح
**الفكرة ببساطة:**
نرسم قيم التشابه بين الجمل؛ ستشاهد بالعين المجردة كيف ينخفض الخط فجأة عند الانتقال من موضوع لآخر (النفق الدلالي / الوادي)، وهو المكان الذي يقطع عنده المشرط الذكي!

In [ ]:
import matplotlib.pyplot as plt

# plot consecutive similarity scores
pairs = [f"S{i+1} - S{i+2}" for i in range(len(similarities))]

plt.figure(figsize=(7, 3.5))
plt.plot(pairs, similarities, marker='o', color='#2563EB', linewidth=2.5, markersize=8)
plt.axhline(y=threshold, color='#DC2626', linestyle='--', label=f'Threshold ({threshold})')
plt.title("Semantic Similarity Curve Across Sentences", fontsize=12, fontweight='bold')
plt.ylabel("Cosine Similarity", fontsize=10)
plt.xlabel("Sentence Pairs", fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

print("المنطقة الواقعة أسفل الخط المتقطع الأحمر هي نقطة القطع التلقائي (Breakpoint)!")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب رسم منحنى التشابه الدلالي بنفسك هنا):



## 🎯 ملخص المعمل السريع (Lecture 37):
1. **التقطيع العادي أعمى:** يقطع بعدد الحروف، فيمزق الجمل أو يدمج موضوعات غير مرتبطة مما يشوه الـ Embeddings.
2. **التقطيع الدلالي ذكي:** يقطع عند تغير الفكرة والمعنى، مما ينتج قطعا مستقلة (`Self-contained`) ونقية وغنية بالسياق.
3. **الخطوات الأربع:** تقسيم لجمل ⬅️ حساب المتجهات ⬅️ قياس التشابه المتجاور ⬅️ القطع عند هبوط التشابه عن العتبة (`Threshold`).
4. **المحاضرة القادمة (Lecture 38):** بناء محرك `Semantic Chunker` كامل بلغة بايثون بدون أي مكتبات خارجية!